# Decision Trees

Coursework study extract, week 5.

Study tree depth, decision regions, splitting criteria, probabilities, and training-only grid search on synthetic moons. The original depth-summary table was unfinished and is not represented as completed. Earlier comparisons inspect the same test split: treat these as exploratory demonstrations, not an untouched final benchmark.

Adapted from completed code in `MCAI500_M5_In_Class_Activity.ipynb`. Instructor-provided scaffolding and pair-work context are acknowledged; authorship of individual lines cannot be independently separated. See [ATTRIBUTION.md](../ATTRIBUTION.md). Teaching prompts, supplied solutions, saved outputs, and identifying metadata are omitted.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

X, y = make_moons(n_samples=600, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=15)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Nonlinear moons dataset")
plt.show()


In [ ]:
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, r2_score

depths = [1, 2, 3, 5, None]
rows = []

for depth in depths:
    model = DecisionTreeClassifier(
        criterion = "gini",
        max_depth = depth,
        random_state = 42
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("Test accuracy:", round(accuracy_score(y_test, y_pred), 3))
    print("Tree depth:", model.get_depth())
    print("Number of leaves:", model.get_n_leaves())
    print()
    pass


In [ ]:
best_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)
best_model.fit(X_train, y_train)

x1_values = np.linspace(
    X_train[:, 0].min() - 0.5,
    X_train[:, 0].max() + 0.5,
    300
)
x2_values = np.linspace(
    X_train[:, 1].min() - 0.5,
    X_train[:, 1].max() + 0.5,
    300
)

xx, yy = np.meshgrid(x1_values, x2_values)

grid = np.c_[xx.ravel(), yy.ravel()]
predictions = best_model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(9, 6))
plt.contourf(
    xx, yy, predictions,
    levels=[-0.5, 0.5, 1.5],
    cmap="coolwarm",
    alpha=0.25
)

plt.scatter(
    X_train[:, 0],
    X_train[:, 1],
    c=y_train,
    cmap="coolwarm",
    edgecolors="k",
    s=25
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Decision Tree decision boundary — max_depth=5")
plt.show()


In [ ]:
gini_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,
    random_state=42
)

entropy_model = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=4,
    random_state=42
)

rows = []

for name, model in {
    "Gini": gini_model,
    "Entropy": entropy_model
}.items():
    model.fit(X_train, y_train)

    rows.append({
        "Criterion": name,
        "Test accuracy": accuracy_score(
            y_test, model.predict(X_test)
        ),
        "Number of leaves": model.get_n_leaves(),
        "Actual depth": model.get_depth()
    })

criterion_results = pd.DataFrame(rows)
display(criterion_results)


In [ ]:
observation = X_test[0:1]

prediction = best_model.predict(observation)
probabilities = best_model.predict_proba(observation)

print("Observation:", observation[0])
print("Actual class:", y_test[0])
print("Predicted class:", prediction[0])

display(pd.DataFrame(
    probabilities,
    columns=[f"P(class {label})" for label in best_model.classes_]
))


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [2, 3, 4, 5, None],
    "min_samples_leaf": [1, 3, 8, 15],
    "max_leaf_nodes": [None, 6, 10, 15]
}

grid_search = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=1
)

grid_search.fit(X_train, y_train)

print("Best settings:", grid_search.best_params_)
print(f"Best mean CV accuracy: {grid_search.best_score_:.2%}")

tuned_model = grid_search.best_estimator_
print(f"Test accuracy: {tuned_model.score(X_test, y_test):.2%}")


## Review notes

Study tree depth, decision regions, splitting criteria, probabilities, and training-only grid search on synthetic moons. The original depth-summary table was unfinished and is not represented as completed. Earlier comparisons inspect the same test split: treat these as exploratory demonstrations, not an untouched final benchmark.

These are learning experiments, not production systems. Scores depend on the data split and package versions. Generated commentary in this publication is editorial context added during preparation, not a claim that it was part of the original class submission.